<a href="https://colab.research.google.com/github/olanigan/homestead/blob/main/examples/colab/01_quickstart_homestead_lfm2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homestead Quickstart on Colab T4

Serve a Liquid **LFM2.5** small language model through **Homestead** (this project's
OpenAI-compatible local-LLM gateway) on a free Colab T4 GPU, and drive one real
tool-calling round trip against it.

**What this notebook proves:** Homestead's own gateway (or a llama.cpp fallback, if
the gateway build fails) can serve `LiquidAI/LFM2.5-1.2B-Instruct-GGUF` on a free T4,
respond on `/v1/chat/completions`, and correctly call a tool when asked.

**Colab/T4 caveats -- read before running:**
- Free-tier T4 sessions disconnect after roughly 90 minutes idle and cap out after a
  few hours of connected time -- this isn't a notebook to leave running unattended.
- Colab's disk is ephemeral: every fresh run re-clones this repo and re-downloads the
  model (a 1.2B GGUF is a couple hundred MB -- a minute or two, not more).
- T4 is a Turing-generation GPU: no native bf16. Everything here runs fp16/int4,
  which is what the GGUF quantizations already are, so this isn't a limitation in
  practice.
- **Runtime -> Change runtime type -> T4 GPU** before running anything below.

Series: this is notebook **01** of the Homestead Colab showcase series -- see
[`examples/colab/README.md`](./README.md) for **02** (Model Battle Arena) and the
roadmap for 03-07 (framework showdown, speculative decoding, vision agents, and more).


In [21]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


## 1. Clone this repo and load the shared helpers

`colab_common.py` (in this same directory) holds the gateway-build-with-fallback
logic shared by every notebook in this series -- see its docstrings for what each
function actually does and why.


In [22]:
import os
import subprocess
import sys

if not os.path.isdir("/content/homestead"):
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1",
         "https://github.com/olanigan/homestead.git",
         "/content/homestead"],
        check=True,
    )

subprocess.run(["pip", "install", "-q", "openai", "huggingface_hub", "requests"], check=True)

sys.path.insert(0, "/content/homestead/examples/colab")
from colab_common import try_build_gateway, serve_model, pick_and_download_gguf


## 2. Try building the real Homestead gateway, fall back to llama.cpp if it fails

This is a genuine attempt, not a demo of failure: if this repo builds cleanly and the
model gets registered, the rest of this notebook talks to Homestead's real gateway
(`homestead provider start`, fixed port `:3030`). If anything in that chain doesn't
work in a fresh Colab VM, `serve_model()` below detects it automatically (an empty
`/v1/models` response) and starts a plain `llama-cpp-python` server instead --
either way you get a real OpenAI-compatible endpoint to talk to.


In [33]:
# Removed explicit PATH modification and preflight check as 'llama' will be moved to /usr/local/bin
print("'llama.cpp' will be moved to /usr/local/bin for global access.")

'llama.cpp' will be moved to /usr/local/bin for global access.


In [24]:
gateway_ok = try_build_gateway()
print("Gateway build:", "OK" if gateway_ok else "skipped/failed -- will try it anyway and fall back if unhealthy")


Gateway build: OK


## 3. Download the model

`LiquidAI/LFM2.5-1.2B-Instruct-GGUF` -- Liquid's 1.2B edge-tuned instruct model, a
good balance of "small enough to be interesting" and "reliable enough at tool-calling
to not need debugging on a first run" -- sub-1B models are demonstrably less reliable
at tool-calling (see notebook 02's battle results for exactly how much less).


In [25]:
MODEL_REPO = "LiquidAI/LFM2.5-1.2B-Instruct-GGUF"
model_path = pick_and_download_gguf(MODEL_REPO)
print(model_path)


LiquidAI/LFM2.5-1.2B-Instruct-GGUF: picked LFM2.5-1.2B-Instruct-Q4_K_M.gguf (out of 8 .gguf files)
/content/models/LFM2.5-1.2B-Instruct-Q4_K_M.gguf


In [38]:
print("Installing llama.cpp...")
!curl -LsSf https://llama.app/install.sh | sh
# Verify source path before moving
print("Verifying ~/.llama-app/llama exists before move:")
!ls -l ~/.llama-app/llama || echo '~/.llama-app/llama not found before move'
# Move the llama executable to a universally accessible PATH location
!mv ~/.llama-app/llama /usr/local/bin/llama
# Verify destination path and global availability after moving
print("Verifying /usr/local/bin/llama exists after move:")
!ls -l /usr/local/bin/llama || echo '/usr/local/bin/llama not found after move'
print("Checking 'llama' in PATH after move:")
!which llama || echo 'llama not found in system PATH'
print("llama.cpp installation and move complete.")

Installing llama.cpp...
Version: b11160
Probing CUDA...
Found: 75
Installation completed successfully

To make llama available in future sessions, add ~/.local/bin to your PATH by running:

  echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bash_profile

Then open a new terminal and run:

  llama serve

To start it now without modifying your PATH, run:

  ~/.llama-app/llama serve

Verifying ~/.llama-app/llama exists before move:
-rwxr-xr-x 1 root root 540487480 Sep 25 22:11 /root/.llama-app/llama
Verifying /usr/local/bin/llama exists after move:
-rwxr-xr-x 1 root root 540487480 Sep 25 22:11 /usr/local/bin/llama
Checking 'llama' in PATH after move:
/usr/local/bin/llama
llama.cpp installation and move complete.


With `llama.cpp` installed, please re-run the following cells to restart the model server and attempt the tool-calling round trip again:

*   **Cell `b3HV_2J0dvzH`**: `serve_model(...)`
*   **Cell `08QHXGgUdvzH`**: `client.chat.completions.create(...)`

## 4. Start serving

`serve_model()` tries the gateway first (if step 2 built it), verifies it actually
reports a healthy, registered model, and falls back to llama.cpp automatically if
not -- see `colab_common.py` for the exact health-check logic.


In [ ]:
import os
import subprocess

# Clean up any stale processes occupying ports 3030 or 8080
print("Cleaning up existing processes on ports 3030 and 8080...")
subprocess.run("fuser -k 3030/tcp || true", shell=True)
subprocess.run("fuser -k 8080/tcp || true", shell=True)

proc, base_url, models, used_gateway = serve_model(
    model_path,
    gateway_ok # Restore gateway_ok to use the now-configured Homestead binary!
)
assert models, (
    "No server came up healthy. Check /content/gateway.log and "
    "/content/llamacpp-8080.log for what went wrong."
)
print(f"Serving via {'Homestead gateway' if used_gateway else 'llama.cpp fallback'} at {base_url}")
print(models)

## 5. One real tool-calling round trip

Ask the model something it can only answer correctly by calling a tool, and confirm
it actually does.


In [41]:
import os
from openai import OpenAI

if 'models' not in globals() or models is None:
    print("ERROR: The model server is not running (models is None).")
    print("--- Checking gateway.log ---")
    if os.path.exists("/content/gateway.log"):
        !cat /content/gateway.log
    else:
        print("gateway.log not found.")
    print("\n--- Checking llamacpp-8080.log ---")
    if os.path.exists("/content/llamacpp-8080.log"):
        !cat /content/llamacpp-8080.log
    else:
        print("llamacpp-8080.log not found.")
else:
    client = OpenAI(base_url=base_url, api_key="none")
    MODEL_ID = models["data"][0]["id"]
    print("Talking to model id:", MODEL_ID)

    tools = [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }]

    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": "What's the weather in Lagos right now?"}],
            tools=tools,
        )
        print(resp.choices[0].message)
    except Exception as e:
        print(f"An error occurred during API call: {e}")

ERROR: The model server is not running (models is None).
--- Checking gateway.log ---

  Starting Homestead provider API on http://localhost:3030/v1...
  API:    http://localhost:3030/v1
  Health: http://localhost:3030/v1/health
  Models: http://localhost:3030/v1/models

  Pi integration:
    cp -r /content/homestead/pi-extension ~/.pi/agent/extensions/homestead
    # Then restart Pi to pick up the extension

Failed to start server. Is port 3030 in use?

--- Checking llamacpp-8080.log ---
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
llama_model_loader: loaded meta data with 34 key-value pairs and 148 tensors from /content/models/LFM2.5-1.2B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = lfm2
llama_model_loader:

## What's next

- **[`02_model_battle_arena.ipynb`](./02_model_battle_arena.ipynb)** -- same serving
  pattern, but sweeps five models (350M through an 8B-A1B MoE, plus one non-Liquid
  competitor) through five real tool-calling tasks and produces a pass-rate
  leaderboard.
- See [`examples/colab/README.md`](./README.md) for the rest of the planned series
  (03-07: framework showdown, speculative-decoding speed, vision agents, thinking vs.
  instant, and a small-model-vs-big-API-model comparison).

Stop the server before closing this notebook so the T4 is free for whatever runs next:


In [ ]:
proc.terminate()
print("Server stopped.")
